In [255]:
import pandas as pd
import numpy as np

In [256]:
df = pd.read_excel(
    r"C:\Users\Saroka\OneDrive\Desktop\Mosca\Asistencia 2025\Asistencia Mosca .xlsx",
    sheet_name='ASISTENCIA 2025',
    skiprows=7
)


In [257]:
df.rename(columns={'Unnamed: 0': 'id_cena'}, inplace=True)

In [258]:
df = df.loc[0:43]
df

,id_cena,Fecha,Axel Zielonka,Federico Saroka,Federico Cabelli,Federico Koltan,Manuel Hirsch,Joaquín Sokolowicz,Lucas Rotmitrovsky,Gonzalo Borinsky,...,Ianick Izon,Santiago Tabak,QUORUM?,Hubo Cena?,Comidas,Categoria comida,Precio,Casa,Postre,Tema de la noche
0,45.0,2025-03-01 00:00:00,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NO,SI,Pizza y Patitas,Patitas y Papas,10000,Cabelli,NaN,Interno / Catan
1,46.0,2025-03-08 00:00:00,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,SI,SI,Empanadas,Empanadas,13500,Saroka,NaN,Merienda / Poker
2,47.0,2025-03-15 00:00:00,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,SI,SI,Bondiola al disco,Disco,11500,Cabelli,NaN,Bondiola
3,48.0,2025-03-22 00:00:00,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,SI,SI,Patitas y Papas,Patitas y Papas,13500,Saroka,NaN,Play / Pelicula
4,49.0,2025-03-29 00:00:00,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,SI,SI,Hamburguesas,Hamburguesas,12800,Manu,NaN,Catan / Los Simuladores
5,50.0,2025-04-05 00:00:00,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,SI,SI,Asado,Asado,15400,Saroka,NaN,Brainrot Italiano
6,51.0,2025-04-12 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN
7,52.0,2025-04-19 00:00:00,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,SI,SI,Brisquet,Asado,18000,Soko,NaN,Brisquet 12 hs / Tateti fútbol
8,53.0,2025-04-26 00:00:00,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,SI,SI,Asado,Asado,NaN,Soko,NaN,Cumple de Soko
9,54.0,2025-05-03 00:00:00,1.0,0.0,0.5,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,SI,SI,Ravioles con bondiola,Pastas,10500,Axel cdc,NaN,Falta drama en el grupo


In [259]:
# 1) columnas en minúscula
df.columns = df.columns.str.lower().str.strip()

# 2) limpiar TODOS los strings: lower + strip + colapsar espacios múltiples
for col in df.select_dtypes(include="object").columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)  # convierte "fede   saroka" -> "fede saroka"
    )


In [260]:
df["id_cena"] = df["id_cena"].astype("int64")
df["fecha"] = pd.to_datetime(df["fecha"]).dt.normalize()


In [261]:
personas = [
    "axel zielonka",
    "federico saroka",
    "federico cabelli",
    "federico koltan",
    "manuel hirsch",
    "joaquín sokolowicz",
    "lucas rotmitrovsky",
    "gonzalo borinsky",
    "matias nemirovsky",
    "ianick izon",
    "santiago tabak"
]

df_long = df.melt(
    id_vars=["id_cena"],
    value_vars=personas,
    var_name="nombre",
    value_name="asistio"
)

# quedarnos con 1 y 0.5
cena_participantes = (
    df_long
    .loc[df_long["asistio"].isin([1, 0.5]), ["id_cena", "nombre", "asistio"]]
    .assign(
        nombre=lambda x: x["nombre"].str.strip().str.lower(),
        extras=lambda x: x["asistio"].apply(
            lambda v: "se fue antes" if v == 0.5 else ''
        )
    )
    .drop(columns="asistio")
)

# orden: primero por id_cena, y dentro por el orden original de personas
orden_personas = {n: i for i, n in enumerate(personas)}
cena_participantes["orden_persona"] = cena_participantes["nombre"].map(orden_personas)

cena_participantes = (
    cena_participantes
    .sort_values(["id_cena", "orden_persona"], ascending=[True, True])
    .drop(columns="orden_persona")
    .reset_index(drop=True)
)


In [262]:
cena_participantes

,id_cena,nombre,extras
0,45,axel zielonka,
1,45,federico saroka,
2,45,federico cabelli,
3,46,axel zielonka,
4,46,federico saroka,
...,...,...,...
280,88,manuel hirsch,
281,88,joaquín sokolowicz,
282,88,lucas rotmitrovsky,
283,88,gonzalo borinsky,


In [263]:
# --- Sheet nueva (ida/vuelta) ---
df_iv = pd.read_excel("Data asistencia Mosca.xlsx", sheet_name="Ubicación ")

# Normalizar SOLO nombres de columnas de la sheet nueva
df_iv.columns = (
    df_iv.columns.astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Renombrar columna de fecha
if "id cena" in df_iv.columns:
    df_iv = df_iv.rename(columns={"id cena": "fecha"})
elif "id_cena" in df_iv.columns:
    df_iv = df_iv.rename(columns={"id_cena": "fecha"})

# Fecha como date (yyyy-mm-dd)
df_iv["fecha"] = pd.to_datetime(df_iv["fecha"]).dt.date

# Marcar 1ra fila por fecha como ida y 2da como vuelta
df_iv = df_iv.sort_values("fecha").copy()
df_iv["tipo"] = df_iv.groupby("fecha").cumcount().map({0: "ida", 1: "vuelta"})
df_iv = df_iv[df_iv["tipo"].isin(["ida", "vuelta"])]

# Long -> Pivot (fecha, nombre, ida, vuelta)
person_cols = [c for c in df_iv.columns if c not in ["fecha", "tipo"]]

iv_long = df_iv.melt(
    id_vars=["fecha", "tipo"],
    value_vars=person_cols,
    var_name="nombre",
    value_name="valor"
)

# Normalizar SOLO los nombres (por si alguna columna venía rara)
iv_long["nombre"] = (
    iv_long["nombre"].astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

iv_pivot = (
    iv_long.pivot_table(
        index=["fecha", "nombre"],
        columns="tipo",
        values="valor",
        aggfunc="first"
    )
    .reset_index()
)

# Asegurar columnas ida/vuelta
if "ida" not in iv_pivot.columns:
    iv_pivot["ida"] = np.nan
if "vuelta" not in iv_pivot.columns:
    iv_pivot["vuelta"] = np.nan

# --- Agregar fecha a cena_participantes para poder mergear ---
df["fecha"] = pd.to_datetime(df["fecha"]).dt.date
iv_pivot["fecha"] = pd.to_datetime(iv_pivot["fecha"]).dt.date

cp = cena_participantes.merge(df[["id_cena", "fecha"]], on="id_cena", how="left")

# --- Merge por fecha + nombre ---
cp = cp.merge(iv_pivot[["fecha", "nombre", "ida", "vuelta"]], on=["fecha", "nombre"], how="left")

# Limpiar columnas ida/vuelta + pasar a minúscula
for col in ["ida", "vuelta"]:
    cp[col] = (
        cp[col]
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace("nan", np.nan)
    )

# --- Regla: antes del 17/05/2025 => '-' ---
cutoff = pd.to_datetime("2025-05-17").date()
mask_old = cp["fecha"] < cutoff
cp.loc[mask_old, ["ida", "vuelta"]] = "-"

# Si no hubo match, también '-'
cp["ida"] = cp["ida"].fillna("-")
cp["vuelta"] = cp["vuelta"].fillna("-")

# --- Output final sin fecha (incluye extras) ---
# cena_participantes ya trae 'extras' por los 0.5
cena_participantes_iv = cp[["id_cena", "nombre", "extras", "ida", "vuelta"]].reset_index(drop=True)


In [ ]:
#cena_participantes_iv.to_excel("cena_participantes.xlsx", index=False)

In [265]:
# tabla base con las columnas que necesitas (ajustá nombres si en tu df están distinto)
cena = df[["id_cena", "quorum?", "hubo cena?", "comidas", "precio", "casa", "tema de la noche", "postre"]].copy()

# renombrar a lo que querés en el output
cena = cena.rename(columns={
    "quorum?": "quorum",
    "hubo cena?": "realizada",
    "comidas": "comida",
    "tema de la noche": "tema"
})

# --- cantidad de personas: contar 1 y 0.5 como 1 persona ---
# (usa df_long, porque ahí está el valor original 0/0.5/1)
cantidad_personas = (
    df_long.loc[df_long["asistio"].isin([1, 0.5])]
    .groupby("id_cena")
    .size()
    .reset_index(name="cantidad personas")
)

cena = cena.merge(cantidad_personas, on="id_cena", how="left")

# si no hay registros, poner 0
cena["cantidad personas"] = cena["cantidad personas"].fillna(0).astype(int)

# --- regla: si realizada == "no" => lo demás en blanco (pero mantener "-" si existe) ---
cols_blank = ["comida", "precio", "casa", "tema", "postre"]

mask_no = cena["realizada"].astype(str).str.strip().str.lower().eq("no")

for col in cols_blank:
    cena.loc[mask_no & (cena[col] != "-"), col] = ""

# también para cantidad personas: si no se realizó, que quede en blanco (pero si fuese "-", mantener)
cena.loc[mask_no & (cena["cantidad personas"].astype(str) != "-"), "cantidad personas"] = ""

# ordenar columnas final exacto como pediste
cena = cena[[
    "id_cena",
    "quorum",
    "realizada",
    "comida",
    "precio",
    "casa",
    "tema",
    "cantidad personas",
    "postre"
]].reset_index(drop=True)


C:\Users\Saroka\AppData\Local\Temp\ipykernel_19412\507412079.py:35: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cena.loc[mask_no & (cena["cantidad personas"].astype(str) != "-"), "cantidad personas"] = ""


In [266]:
# --- POSTRE ---
mask_old = cena["id_cena"] < 58
mask_new = cena["id_cena"] >= 58

# asegurarse que sea string comparable
cena["postre"] = cena["postre"].astype(str).str.strip().str.lower()

# reglas
cena.loc[mask_old & (cena["postre"] == "nan"), "postre"] = pd.NA
cena.loc[mask_new & (cena["postre"] == "nan"), "postre"] = "no hubo"
cena.loc[mask_new & (cena["postre"] == "-"), "postre"] = "no hubo"

# --- PRECIO ---
cena["precio"] = (
    cena["precio"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace("nan", pd.NA)
)


In [267]:
mask_88 = cena["id_cena"] == 88

cena.loc[mask_88, "precio"] = (
    cena.loc[mask_88, "precio"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace(".00", "", regex=False)
    .replace("nan", pd.NA)
    .astype("Int64")
)


In [268]:
cena['precio'].replace(pd.NA,0,inplace=True)
cena['precio'].replace('-',0,inplace=True)
cena['precio'].replace('',pd.NA,inplace=True)

C:\Users\Saroka\AppData\Local\Temp\ipykernel_19412\2953492950.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cena['precio'].replace(pd.NA,0,inplace=True)
C:\Users\Saroka\AppData\Local\Temp\ipykernel_19412\2953492950.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

In [269]:
cena["precio"] = pd.to_numeric(cena["precio"], errors="coerce").astype("Int64")



In [270]:
cena['precio'].astype('Int64')

0     10000
1     13500
2     11500
3     13500
4     12800
5     15400
6      <NA>
7     18000
8         0
9     10500
10    12000
11    12000
12     <NA>
13    10400
14    20000
15    20000
16    16000
17    13000
18     7800
19     8800
20    42300
21    17000
22    22500
23    10000
24    10000
25    14600
26    15000
27    21000
28    17000
29    17750
30    19000
31    21000
32    16000
33        0
34     <NA>
35    12500
36    10000
37     <NA>
38    18000
39    20000
40    21000
41    11000
42     <NA>
43    15000
Name: precio, dtype: Int64

In [ ]:
#cena.to_excel("cena.xlsx", index=False)

In [272]:
# --- universo: todas las combinaciones id_cena x persona ---
df_personas = pd.DataFrame({"nombre": personas})

universo = (
    cena[["id_cena", "realizada"]]
    .assign(_k=1)
    .merge(df_personas.assign(_k=1), on="_k")
    .drop(columns="_k")
)

# --- 1) Cenas NO realizadas: incluir a todos con razon = "no hubo cena" ---
inas_no = (
    universo[universo["realizada"].astype(str).str.strip().str.lower().eq("no")]
    .loc[:, ["id_cena", "nombre"]]
    .assign(razon="no hubo cena")
)

# --- 2) Cenas realizadas: incluir solo los que NO asistieron, razon = NA (por ahora) ---
asis = cena_participantes[["id_cena", "nombre"]].copy()
asis["nombre"] = asis["nombre"].astype(str).str.strip().str.lower().str.replace(r"\s+", " ", regex=True)

uni_si = universo[universo["realizada"].astype(str).str.strip().str.lower().eq("si")].copy()

inas_si = (
    uni_si.merge(asis, on=["id_cena", "nombre"], how="left", indicator=True)
    .query('_merge == "left_only"')
    .drop(columns=["_merge", "realizada"])
    .assign(razon=pd.NA)
)

# --- unir todo ---
inasistencia = (
    pd.concat([inas_no, inas_si], ignore_index=True)
    .sort_values(["id_cena", "nombre"])
    .reset_index(drop=True)
)

# inasistencia queda con: id_cena | nombre | razon


In [274]:
inasistencia.to_excel('inasistencia.xlsx', index=False)

In [308]:
casas = pd.read_csv('casa.csv')
cena_participantes_iv = pd.read_excel("cena_participantes.xlsx")
cena = pd.read_excel("cena.xlsx")

In [309]:
def norm(s):
    return (
        s.astype(str)
         .str.lower()
         .str.strip()
         .str.replace(r"\s+", " ", regex=True)
    )


In [310]:
u_ida = norm(cena_participantes_iv["ida"])
u_vuelta = norm(cena_participantes_iv["vuelta"])
u_casa = norm(cena["casa"])

In [311]:
lugares_unicos = pd.concat([u_ida, u_vuelta, u_casa]).dropna().unique()
lugares_unicos

array(['-', 'axel capi', 'saroka', 'cabelli', 'manu', 'nemi', 'ianick',
       'axel cdc', 'sheila', 'club', 'soko', 'niki', 'gonzi cdc',
       'ezeiza', 'tabak cdc', 'koltan cdc', 'cardales', 'sauna',
       'campo soko', 'freddo', 'gym cdc', 'bungalows', 'caro',
       'martu szkol', 'gonzi capi', 'koltan capi', 'village', 'nan',
       'bataxa', 'sinchi'], dtype=object)

In [312]:
casas_csv = norm(casas["casa"]).dropna().unique()


In [313]:
faltantes = sorted(set(lugares_unicos) - set(casas_csv))


In [314]:
df_faltantes = pd.DataFrame({"casa_faltante": faltantes})
df_faltantes


,casa_faltante
0,-
1,nan


In [321]:
cena_participantes_iv

,id_cena,nombre,extras,ida,vuelta
0,45,axel zielonka,NaN,-,-
1,45,federico saroka,NaN,-,-
2,45,federico cabelli,NaN,-,-
3,46,axel zielonka,NaN,-,-
4,46,federico saroka,NaN,-,-
...,...,...,...,...,...
281,88,manuel hirsch,NaN,manu,manu
282,88,joaquín sokolowicz,NaN,soko,soko
283,88,lucas rotmitrovsky,NaN,-,-
284,88,gonzalo borinsky,NaN,gonzi cdc,gonzi cdc


In [ ]:
#cena_participantes_iv.to_excel("cena_participantes.xlsx", index=False)

In [9]:
df = pd.read_excel("Asistencia Mosca .xlsx", sheet_name='ASISTENCIA 2025', skiprows=7)

In [18]:
df=df.loc[0:43]

In [19]:
df["Fecha "] = pd.to_datetime(df["Fecha "]).dt.normalize()

C:\Users\Saroka\AppData\Local\Temp\ipykernel_11032\3504046330.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Fecha "] = pd.to_datetime(df["Fecha "]).dt.normalize()


In [24]:
# df es tu tabla principal (id_cena, fecha, ...)
cena = cena.merge(
    df[["id_cena", "fecha"]],
    on="id_cena",
    how="left"
)

# asegurar formato YYYY-MM-DD
cena["fecha"] = pd.to_datetime(cena["fecha"]).dt.date


In [6]:
casas = pd.read_csv('casa.csv')
cena = pd.read_excel("cena.xlsx")

In [ ]:
#cena.to_excel('cena.xlsx')

In [235]:
df_cena['comida'] = df_cena['comida'].replace('-', np.nan)

In [223]:
df_cena

,id_cena,fecha,quorum,comida,precio,casa,tema
0,1,2024-03-02,SI,Glorias,NaN,Glorias,NaN
1,2,2024-03-09,NO,Asado,NaN,Cabelli,NaN
2,3,2024-03-16,SI,Asado,9500.00,Saroka,NaN
3,4,2024-03-23,SI,Glorias,8500.00,Glorias,NaN
4,5,2024-03-30,SI,Asado,12000.00,Soko,NaN
5,6,2024-04-06,SI,Pizzas y empanadas,8000.00,Gonzi cdc,NaN
6,7,2024-04-13,SI,Asado,9500.00,Romi nor,Make it memes
7,8,2024-04-20,SI,Cosco,8500.00,Saroka,Make it memes
8,9,2024-04-27,SI,Sinchi,7800.00,Tabak cdc,NaN
9,10,2024-05-04,NO,-,NaN,-,-
